In [2]:
import pandas as pd

# Ensure you required the premission to access the DrugBank data and have the CSV file in the correct path.
df = pd.read_csv(r"drugbank_data.csv")

In [3]:
import re
def process_absorption_text(text):
    text = re.sub(r'<[^>]*>', '', text)
    sentences = re.split(r'(?<=[a-zA-Z])\.', text)
    return sentences

### Stage 1:  Implicit Extraction Prior


In [ ]:
# import openai  # Or your preferred LLM library
from mistralai import Mistral
client = Mistral(api_key="Your-API-Key-Here")
def extract_pk(text):
    # This prompt defines the 'Chain of Thought' steps
    user_prompt = f"""
        Task: You will receive a parameter mention from scientific literature. Your task is to accurately extract the PK parameter name, its value, and the unit of measurement.

        - Accuracy is paramount. If the text does not precisely refer to any of the concepts listed below, please answer NIL.
        - If the text does not refer to a PK parameter (e.g. part of a pharmacodynamic parameter, or a PBPK, chemical, in vitro, clinical, or other unrelated term), please answer NIL.
        - If you are unsure, please answer NIL.

        Text: ``{text}''

        Follow the steps above and output the final data row.
    """
    model = "mistral-small"
    # Example call to an LLM API (e.g., GPT-4 or Gemini)
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "system", "content": "You are an expert clinical data extraction specialist. Extract Pharmacokinetic (PK) data from the text and output it strictly as a single CSV-formatted row. The columns must be in this exact order: 'PK parameter', 'value', 'unit'. Do not include headers. Do not include any introductory or concluding text. Output ONLY the comma-separated values. Example output: 'Cmax,15.2,mg/L \n'"},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.5,
        max_tokens=300,
    )
    return response.choices[0].message.content


### Stage 2: Rule-based Structural Extraction

This approach encourages restrict the model to extract following rules.

In [ ]:
# import openai  # Or your preferred LLM library
from mistralai import Mistral
client = Mistral(api_key="Your-API-Key-Here")
def extract_pk(text):
    # This prompt defines the 'Chain of Thought' steps
    user_prompt = f"""
        Task: Extract Pharmacokinetic (PK) data from the text provided below.

        You possess the skill to extract pharmacokinetic (PK) parameters from text. Apply this skill to the following sentence by strictly adhering to these extraction rules:

        **PK Extraction Rules:**
        1.  A PK parameter is identified as a sequence: [Descriptive Name] ... [Numerical Value] [Unit], or,  [Descriptive Name1][Descriptive Name2] ... [Numerical Value1] [Unit1] ... [Numerical Value2] [Unit2].
        2.  Locate all such sequences within the provided sentence. Including also similar sequences that may be part of a larger phrase.
        3.  For each sequence, identify the 'PK parameter' as the descriptive name, the 'Value' as the numerical value, and the 'Unit' as the unit of measurement.
        4.  Structure the output as a CSV-formatted row with columns: 'PK parameter', 'value', 'unit'.

        Only include values explicitly mentioned. If you are unsure, return NIL only.

        Text: "{text}"

        Follow the steps above and output the final data row.
    """
    model = "mistral-small"
    # Example call to an LLM API (e.g., GPT-4 or Gemini)
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "system", "content": "You are an expert clinical data extraction specialist. Extract Pharmacokinetic (PK) data from the text and output it strictly as a single CSV-formatted row. The columns must be in this exact order: 'PK parameter', 'value', 'unit'. Do not include headers. Do not include any introductory or concluding text. Output ONLY the comma-separated values. Example output: 'Cmax,15.2,mg/L \n'"},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.5,
        max_tokens=300,
    )
    return response.choices[0].message.content


### Stage 3: Procedural Reasoning

This approach encourages the model to verbalize its reasoning process step-by-step before providing the final answer. This can lead to more accurate and robust extractions, especially for complex sentences.

In [5]:
# import openai  # Or your preferred LLM library
from mistralai import Mistral
client = Mistral(api_key="Your-API-Key-Here")
def extract_pk(text):
    # This prompt defines the 'Chain of Thought' steps
    user_prompt = f"""
        Task: Extract Pharmacokinetic (PK) data from the text provided below.

        1.  Step 1: Identify the drug name.
        2.  Step 2: Identify all routes of administration (e.g., IV bolus, Infusion, SC).
        3.  Step 3: Identify only the oral route; discard the other routes and identify the dose.
        4.  Step 4: Locate PK parameters (Cmax, Tmax, AUC, Bioavailability, etc.) linked to those doses.
        5.  Step 5: Format the final output as a structured format (as Example output).

        Only include values explicitly mentioned. If you are unsure, return NIL only.

        Text: ``{text}''

        Follow the steps above and output the final data row.
    """
    model = "mistral-small"
    # Example call to an LLM API (e.g., GPT-4 or Gemini)
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "system", "content": "You are an expert clinical data extraction specialist. Extract Pharmacokinetic (PK) data from the text and output it strictly as a single CSV-formatted row. The columns must be in this exact order: 'PK parameter', 'value', 'unit'. Do not include headers. Do not include any introductory or concluding text. Output ONLY the comma-separated values. Example output: 'Cmax,15.2,mg/L \n'"},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.5,
        max_tokens=300,
    )
    return response.choices[0].message.content


### Stage 4: CORVA 

CORVA prompting encourages the model to explore multiple reasoning paths and self-correct or combine ideas. Here, it will brainstorm possibilities, evaluate them, and then synthesize the final answer.

In [ ]:
# import openai  # Or your preferred LLM library
from mistralai import Mistral
client = Mistral(api_key="Your-API-Key-Here")
def extract_pk(text):
    # This prompt defines the 'Chain of Thought' steps
    user_prompt = f"""
        Task: Extract Pharmacokinetic (PK) data from the text provided below.

        Follow this Tree of Thoughts process:
        1.  **Brainstorm Potential PK Parameters**: List all possible phrases or segments in the sentence that *might* represent a PK parameter (description, value, unit). Do not filter yet.
        2.  **Evaluate Each Potential Parameter**: For each item brainstormed, critically assess if it truly fits the definition of a PK parameter through the oral route (valid numerical value, and distinct unit). Discard any that are incomplete or ambiguous.
        3.  **Synthesize Final Extraction**: Combine all the *valid* and *reasoned* PK parameters into a structured output as Example. Each element in the array should be an object with 'PK parameter', 'Value', and 'Unit' keys.
        4.  **Check for Conflicts**: Ensure that the extracted unit is consistent with the expected unit for each PK parameter.

        Only include values explicitly mentioned. If you are unsure, return NIL only.

        Text: "{text}"

        Follow the steps above and output the final data row.
    """
    model = "mistral-small"
    # Example call to an LLM API (e.g., GPT-4 or Gemini)
    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "system", "content": "You are an expert clinical data extraction specialist. Extract Pharmacokinetic (PK) data from the text and output it strictly as a single CSV-formatted row. The columns must be in this exact order: 'PK parameter', 'value', 'unit'. Do not include headers. Do not include any introductory or concluding text. Output ONLY the comma-separated values. Example output: 'Cmax,15.2,mg/L \n'"},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.5,
        max_tokens=300,
    )
    return response.choices[0].message.content


### PK Extraction Framework

This framework drives the prompts from each stage. 

In [ ]:
import pandas as pd
import io
import time
from tqdm import tqdm # Optional: adds a progress bar

extracted_data = []

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    # Skip if absorption is empty
    if pd.isna(row['absorption']):
        continue

    sentences = process_absorption_text(row['absorption'])
    
    # Process each sentence with a retry mechanism
    for sentence in sentences:
        max_retries = 5
        retry_count = 0
        wait_time = 2  # Start with 2 seconds
        
        while retry_count < max_retries:
            try:
                # Call your extraction function
                data_string = extract_pk(sentence)
                
                if data_string != 'NIL':
                    # Parse the CSV string into a temporary DF
                    tempDF = pd.read_csv(io.StringIO(data_string), 
                                         names=['PK parameter', 'value', 'unit'])
                    
                    # Attach metadata
                    tempDF['drugbank_id'] = row['drugbank_id']
                    tempDF['name'] = row['name']
                    tempDF['chembl_id'] = row['chembl_id']
                    
                    # Store as dictionaries
                    extracted_data.extend(tempDF.to_dict('records'))
                
                # If successful, break the while loop to move to next sentence
                break 

            except Exception as e:
                # Check if it's a Rate Limit error (Status 429)

                # print(f"\nRate limit hit. Retrying in {wait_time}s...")
                time.sleep(wait_time)
                retry_count += 1
                wait_time *= 2  # Exponential backoff


# Final DataFrame assembly
result_absorption = pd.DataFrame(extracted_data)
# Reorder columns
cols = ['drugbank_id', 'name', 'chembl_id', 'PK parameter', 'value', 'unit']
result_absorption = result_absorption[cols] if not result_absorption.empty else pd.DataFrame(columns=cols)


### Evaluation


In [ ]:
import pandas as pd

df_extract = pd.read_csv(r"results_extracted.csv")
df_gt = pd.read_csv(r"annotated_ground_truth.csv")
time_units = {
    'h', 'hr', 'hour', 'hours',
    'min', 'mins', 'minute', 'minutes',
    'day', 'days',
    'week', 'weeks',
    'month', 'months',
    'year', 'years',
    'time'
}
df_time_only = df_extract[df_extract["unit"].str.lower().isin(time_units)]
df_time_only_final = df_time_only[df_time_only["PK parameter"].str.lower().str.contains(
        r'time|peak|tmax|cmax|absorption|ka',
        na=False
    )].dropna()

df_time_only_final['value'] = df_time_only_final.value.str.extract(r'(\d+[.\d]*)')
df_time_only_final['value'] = pd.to_numeric(df_time_only_final['value'], errors='coerce')

In [ ]:
import pandas as pd
import numpy as np

# Make copies
gt = df_gt.rename(str.lower, axis="columns").copy()
pred = df_time_only_final.copy()

# Normalize columns
for col in ["drugbank_id", "unit"]:
    gt[col] = gt[col].str.lower().str.strip()
    pred[col] = pred[col].str.lower().str.strip()

# Numeric tolerance for value comparison
tolerance = 0.00001

results = []

for _, gt_row in gt.iterrows():
    drug_id = gt_row["drugbank_id"]
    
    # All predictions for this drug
    pred_rows = pred[pred["drugbank_id"] == drug_id]
    
    hit = len(pred_rows) > 0
    unit_map = {
    "h": "hours",
    "hour": "hours",
    "hr": "hours"
    }
    pred["unit"] = pred["unit"].replace(unit_map)

    value_correct = False
    unit_correct = False
    exact_match = False
    
    if hit:

        # Check value match (any prediction close enough)
        value_correct = np.any(
            np.isclose(
                pred_rows["value"],
                gt_row["tmax"],
                atol=tolerance
            )
        )

        
        # Check unit match
        unit_correct = np.any(
            pred_rows["unit"] == gt_row["unit"]
        )
        
        # Exact match (value + unit in same row)
        for _, p in pred_rows.iterrows():
            if (
                np.isclose(p["value"], gt_row["tmax"], atol=tolerance)
                and p["unit"] == gt_row["unit"]
            ):
                exact_match = True
                break

    results.append({
        "drugbank_id": drug_id,
        "hit": hit,
        "value_correct": value_correct,
        "unit_correct": unit_correct,
        "exact_match": exact_match
    })

evaluation = pd.DataFrame(results)
evaluation.to_csv(r"saved_evaluation.csv", index=False) # evaluation saved for later analysis 

In [ ]:
recall = evaluation["hit"].mean()
value_accuracy = evaluation["value_correct"].mean()
unit_accuracy = evaluation["unit_correct"].mean()
exact_accuracy = evaluation["exact_match"].mean()

precision = evaluation["exact_match"].sum() / len(pred)

f1 = 2 * (precision * recall) / (precision + recall)

print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)

print("Value Accuracy:", value_accuracy)
print("Unit Accuracy:", unit_accuracy)
print("Exact Match Accuracy:", exact_accuracy)